# GPU (T4) Method Comparison — Method 1 vs Cascade Hybrid (ResNet18 / EfficientNet-B0)

Every result up to this point in the project used CPU timing (a MacBook M1)
for fast, cheap internal iteration — see `docs/decision_log.md` and
`docs/01_cascade_journey_summary.md`. Those numbers are valid for comparing
methods *against each other* (same hardware for all of them), but the
project's cost report (`CLAUDE.md` Section 7: params, GFLOPs, VRAM, **FPS
on T4**) needs a real GPU measurement, not a CPU one.

**This notebook is atomic and self-contained on purpose**: each step is
its own cell, runs one method, and prints/saves that method's numbers
before moving to the next — so a reviewer can run it cell by cell and see
exactly which method produced which result, in the same order the research
was actually done. Nothing here is a black box; every step calls the same
`scripts/*.py` used throughout the project (config-driven, per
`CLAUDE.md` Section 9 — no notebook-only logic).

**Methods compared (all on the gold test set, all GPU-accelerated):**
1. Plain YOLO26n, zero-shot, conf=0.25 (Method 1 / baseline)
2. Cascade hybrid: YOLO26n (low-conf) + ResNet18 (**standard** 7x7 stem)
   verifier, score fusion — the "before" side of the stem experiment
3. Cascade hybrid: YOLO26n (low-conf) + ResNet18 (**small-input**, 3x3
   stem) verifier, score fusion — item 1b's best CPU-dev result, and the
   "after" side of the same experiment
4. Cascade hybrid: YOLO26n (low-conf) + EfficientNet-B0 verifier, score
   fusion — the alternative backbone tried alongside it

**Before running:** upload `test_frames.zip` (the gold test set's 196
frames — gitignored, regenerable, not in the repo) to
`/content/drive/MyDrive/object-detection/test_frames.zip`. Of the three
verifier checkpoints needed, two should already be on Drive from the
earlier training notebook runs (`resnet18_verifier_small_stem.pt`,
`efficientnet_b0_verifier.pt`); `resnet18_verifier_video_balanced.pt`
(the standard-stem one) was trained locally on CPU rather than via that
notebook, so it needs a one-time manual upload to the same Drive folder.

## Step 0 — Setup: clone the repo, install dependencies, confirm GPU

In [ ]:
import os
if not os.path.exists("object-detection-drone"):
    !git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!git pull origin main --no-edit --no-rebase
!pip install -q -r requirements.txt


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (results below won't be the T4 numbers)")


## Step 1 — Get the gold test set's frames (not tracked in git — regenerable/large)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TEST_FRAMES_ZIP = "/content/drive/MyDrive/object-detection/test_frames.zip"
!unzip -q -o "{TEST_FRAMES_ZIP}" -d .
# zip was created from the repo root, so this restores data/processed/frames/test/ directly
!echo "test frames: $(find data/processed/frames/test -name '*.jpg' | wc -l)"

import json as _json
print("gold annotations boxes:", len(_json.load(open("data/gold_test/annotations.json"))["annotations"]))


## Step 2 — Method 1: plain YOLO26n, zero-shot, conf=0.25

Baseline. YOLO's own checkpoint (`yolo26n.pt`) auto-downloads via
`ultralytics` if not already present — no Drive dependency for this one.
`ultralytics` picks up the GPU automatically when available (no code
change needed vs. the CPU runs).

In [ ]:
!python scripts/05_yolo_zeroshot_eval.py --config configs/05_yolo_zeroshot.yaml


In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/yolo26_zeroshot/predictions.json \
    --method-name yolo26n_zeroshot_T4 \
    --config configs/06_evaluate.yaml


## Step 3 — Cascade hybrid: YOLO26n + ResNet18 (**standard** 7x7 stem) verifier, score fusion

The "before" side of item 1b's stem experiment (`docs/decision_log.md`,
"First and only configuration to beat method 1's own F1"): the standard
ResNet18 stem, video-balanced sampling, no architecture change. Run
alongside the small-input-stem version (next step) so the 3x3-vs-7x7
effect (`docs/01_cascade_journey_summary.md`, Step 8) is shown on real
GPU numbers too, not just the CPU numbers used to discover it.

In [ ]:
config_path = "configs/25_cascade_score_fusion_eval.yaml"
config = yaml.safe_load(open(config_path))
config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
config["verifier_checkpoint"] = "/content/drive/MyDrive/object-detection/resnet18_verifier_video_balanced.pt"
config["output_dir"] = "results/cascade_score_fusion_standard_stem_T4"
yaml.safe_dump(config, open("configs/25_T4.yaml", "w"), sort_keys=False)
print(open("configs/25_T4.yaml").read())


In [ ]:
!python scripts/25_cascade_score_fusion_eval.py --config configs/25_T4.yaml


In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/cascade_score_fusion_standard_stem_T4/predictions.json \
    --method-name yolo26n_score_fusion_standard_stem_T4 \
    --config configs/06_evaluate.yaml


## Step 4 — Cascade hybrid: YOLO26n + ResNet18 (small-input-stem) verifier, score fusion

Item 1b's best CPU-development result (`docs/01_cascade_journey_summary.md`,
Step 8): mAP@.5 0.788, F1 0.791 on CPU. Patches `configs/27` in memory —
`device: cuda` and the verifier checkpoint's Drive path — rather than
editing the file, so the committed config stays the CPU-runnable version
used during development.

In [ ]:
import yaml

config_path = "configs/27_cascade_score_fusion_small_stem.yaml"
config = yaml.safe_load(open(config_path))
config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
config["verifier_checkpoint"] = "/content/drive/MyDrive/object-detection/resnet18_verifier_small_stem.pt"
config["output_dir"] = "results/cascade_score_fusion_small_stem_T4"
yaml.safe_dump(config, open("configs/27_T4.yaml", "w"), sort_keys=False)
print(open("configs/27_T4.yaml").read())


In [ ]:
!python scripts/25_cascade_score_fusion_eval.py --config configs/27_T4.yaml


In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/cascade_score_fusion_small_stem_T4/predictions.json \
    --method-name yolo26n_score_fusion_small_stem_T4 \
    --config configs/06_evaluate.yaml


## Step 5 — Cascade hybrid: YOLO26n + EfficientNet-B0 verifier, score fusion

The alternative backbone (`docs/01_cascade_journey_summary.md`'s
EfficientNet-B0 addendum): best val F1 of the three verifiers (0.9835)
but the *worst* test-set result on CPU (mAP@.5 0.779) — and ~5x slower
per crop than ResNet18 on CPU. This GPU run checks whether that CPU speed
gap (attributed to depthwise-separable convolutions vectorizing poorly on
general CPU kernels) actually disappears on GPU, where such gaps are
expected to shrink or reverse.

In [ ]:
config_path = "configs/33_cascade_score_fusion_efficientnet_b0.yaml"
config = yaml.safe_load(open(config_path))
config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
config["verifier_checkpoint"] = "/content/drive/MyDrive/object-detection/efficientnet_b0_verifier.pt"
config["output_dir"] = "results/cascade_score_fusion_efficientnet_b0_T4"
yaml.safe_dump(config, open("configs/33_T4.yaml", "w"), sort_keys=False)
print(open("configs/33_T4.yaml").read())


In [ ]:
!python scripts/25_cascade_score_fusion_eval.py --config configs/33_T4.yaml


In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/cascade_score_fusion_efficientnet_b0_T4/predictions.json \
    --method-name yolo26n_score_fusion_efficientnet_b0_T4 \
    --config configs/06_evaluate.yaml


## Step 6 — Final comparison table

`results/eval/comparison.csv` now has a `_T4` row for each of the three
methods run above, alongside every CPU-dev row already in it. This is the
authoritative, GPU-measured table for the presentation's cost/quantitative
section.

In [ ]:
import pandas as pd
df = pd.read_csv("results/eval/comparison.csv")
df[df["method"].str.endswith("_T4")]


## Step 7 — Push results back to GitHub

Only the small, text-based artifacts (`metrics.json`, `summary.md`,
`comparison.csv`, manifests, predictions) — no images, no checkpoints
(`*.pt` stays on Drive, gitignored as always).

In [ ]:
import getpass
gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
!git add results/eval results/manifests \
    results/yolo26_zeroshot/predictions.json \
    results/cascade_score_fusion_standard_stem_T4/predictions.json \
    results/cascade_score_fusion_small_stem_T4/predictions.json \
    results/cascade_score_fusion_efficientnet_b0_T4/predictions.json
!git commit -m "Add T4 GPU method comparison (Method 1 vs ResNet standard/small-stem vs EfficientNet cascade)"
!git pull origin main --no-edit --no-rebase
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main
